In [1]:
import pandas as pd

data = pd.read_csv("data/rag_sample_qas_from_kis.csv") 
data.head()

,ki_topic,ki_text,sample_question,sample_ground_truth
0,Setting Up a Mobile Device for Company Email,**Setting Up a Mobile Device for Company Email...,"""How do I set up my company email on my mobile...",To set up your company email on your mobile de...
1,Resetting a Forgotten PIN,**Resetting a Forgotten PIN**\n\nIf you have f...,"I forgot my PIN, how can I reset it?","Don't worry, I'm here to help To reset your fo..."
2,Configuring VPN Access for Remote Workers,**Configuring VPN Access for Remote Workers**\...,How do I set up VPN access on my laptop so I c...,To set up VPN access on your laptop and access...
3,Troubleshooting Issues with Microsoft Office,**Troubleshooting Issues with Microsoft Office...,"""My Microsoft Word keeps freezing every time I...",I'd be happy to help you troubleshoot the issu...
4,Setting Up a Conference Call on Cisco Webex,"To set up a conference call on Cisco Webex, fo...",How do I set up a conference call on Cisco Web...,To set up a conference call on Cisco Webex wit...


In [2]:
text_list = [f"Q: {q}\nA: {a}" for q, a in zip(data["ki_topic"], data["sample_ground_truth"])]

# Save to a Markdown (.md) file
with open("dataset.md", "w", encoding="utf-8") as f:
    f.write("\n\n".join(text_list))

In [3]:
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import openai
from dotenv import load_dotenv
import os
import shutil

load_dotenv()

openai.api_key = os.environ['OPENAI_API_KEY']

CHROMA_PATH = "chroma"

#DATA_PATH = os.path.join(os.getcwd())
Data_path = "dataset.md"

def main():
    generate_data_store()

def generate_data_store():
    documents = load_documents()
    chunks = split_text(documents)
    save_to_chroma(chunks)

def split_text(documents:list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=100,
        length_function = len,
        add_start_index=True,

    )
    chunks = text_splitter.split_documents(documents)
    document = chunks[10]
    #print(f"This is from split_text function : {document}")

    return chunks

def save_to_chroma(chunks:list[Document]):
    if os.path.exists(CHROMA_PATH):
        shutil.rmtree(CHROMA_PATH)

    db = Chroma.from_documents(
        chunks,OpenAIEmbeddings(),persist_directory=CHROMA_PATH
    )
    db.persist()
    print(f"Saved {len(chunks)}")

def load_documents():
    #loader = DirectoryLoader(DATA_PATH, glob="*.md")
    loader = TextLoader(Data_path)
    documents = loader.load()
    print(f"This is from load function : {documents}")
    return documents

if __name__ == "__main__":
    main()


This is from load function : [Document(metadata={'source': 'dataset.md'}, page_content='Q: Setting Up a Mobile Device for Company Email\nA: To set up your company email on your mobile device, please follow these steps:\n\n**First, ensure that you have a supported operating system (iOS, Android, or Windows) and a company email account.**\n\n1. **Check if a Mobile Device Management (MDM) profile is required**: If your company requires MDM for mobile devices, ensure that the profile is installed on your device. If you\'re unsure, contact your IT department for assistance.\n2. **Set up your email account**:\n\t* Go to the Settings app on your mobile device.\n\t* Select "Mail" or "Email" (depending on your device\'s operating system).\n\t* Tap "Add Account" or "Create a new account".\n\t* Select "Exchange" or "Corporate" as the account type.\n\t* Enter your company email address and password.\n\t* If prompted, enter the company\'s email server address (e.g., mail.company.com).\n\t* Select t

C:\Users\aishw\AppData\Local\Temp\ipykernel_21240\2532609604.py:50: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()
